In [0]:
# Cell 1 — Configuration
# Source and target locations for the bronze-to-silver light-cleaning pipeline.

CATALOG = "adventure_works"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

# The 6 tables to copy from bronze to silver (same names, cleaned data).
TABLES = [
    "product",
    "sales_order_detail",
    "sales_territory",
    "purchase_order_detail",
    "purchase_order_header",
    "vendor",
]

# Make sure the target schema exists (idempotent).
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

print(f"Configured: {len(TABLES)} tables, {CATALOG}.{BRONZE_SCHEMA} -> {CATALOG}.{SILVER_SCHEMA}")

In [0]:
# Cell 2 — Load from bronze with light cleaning, write to silver
# Light cleaning = trim leading/trailing whitespace on every string column.
# Written with overwrite mode under the exact same table names.

from pyspark.sql import functions as F

for table_name in TABLES:
    bronze_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}")

    # Detect string columns from the schema and trim only those.
    string_cols = {f.name for f in bronze_df.schema.fields if f.dataType.simpleString() == "string"}
    cleaned_df = bronze_df.select(
        [F.trim(F.col(c)).alias(c) if c in string_cols else F.col(c) for c in bronze_df.columns]
    )

    cleaned_df.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.{table_name}")

    row_count = bronze_df.count()
    print(f"{table_name}: {row_count} rows read, trimmed {len(string_cols)} string columns, written to {CATALOG}.{SILVER_SCHEMA}.{table_name}")

In [0]:
# Cell 3 — Verification check
# Confirm that bronze and silver row counts match for all 6 tables.

counts = []
for table_name in TABLES:
    bronze_rows = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}").count()
    silver_rows = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{table_name}").count()
    counts.append((table_name, bronze_rows, silver_rows))

verification_df = spark.createDataFrame(
    counts, "table_name STRING, bronze_rows LONG, silver_rows LONG"
).withColumn("row_counts_match", F.col("bronze_rows") == F.col("silver_rows"))

display(verification_df)

all_match = verification_df.where("row_counts_match = false").count() == 0
print(f"All 6 tables verified: {all_match}")